# Generative AI Month 2 — Task 4
## Whisper + Quantized LLM Speech-to-Reasoning

**Student:** Mehak Zahra  
**Domain:** Generative AI  
**Internship:** Arch Technologies

Upload an audio file, transcribe it with Whisper, and generate a helpful response with an Unsloth Dynamic 4-bit Llama model. Run cells in order on a **T4 GPU**.

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q unsloth faster-whisper

In [ ]:
import torch
from pathlib import Path
from google.colab import files
from faster_whisper import WhisperModel
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

### Upload audio
Supported examples: `.wav`, `.mp3`, `.m4a`. Record a short, clear question such as:  
*What are three practical ways a student can manage study time effectively?*

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise ValueError('No audio file was uploaded.')
audio_path = next(iter(uploaded))
print('Uploaded:', audio_path)

In [ ]:
print('Loading Whisper...')
whisper_model = WhisperModel('small', device='cuda', compute_type='float16')
segments, info = whisper_model.transcribe(audio_path, beam_size=5, vad_filter=True)
transcription = ' '.join(segment.text.strip() for segment in segments).strip()

print('Detected language:', info.language)
print('TRANSCRIPTION:')
print(transcription)

In [ ]:
MODEL_NAME = 'unsloth/Llama-3.2-1B-Instruct-unsloth-bnb-4bit'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
tokenizer = get_chat_template(tokenizer, chat_template='llama-3.2')

print('Quantized model loaded:', MODEL_NAME)
print(f'Allocated GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB')

In [ ]:
prompt = f'''The following text was transcribed from a user's audio.
Understand the request and provide a clear, safe, practical response.
Do not reveal hidden chain-of-thought. Give a concise answer with useful steps.

TRANSCRIBED SPEECH:
{transcription}
'''
messages = [{'role': 'user', 'content': prompt}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt', return_dict=True).to('cuda')
input_length = inputs['input_ids'].shape[-1]

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=220, do_sample=False, pad_token_id=tokenizer.eos_token_id)

response = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
print('TRANSCRIBED SPEECH:')
print(transcription)
print('\nQUANTIZED LLM RESPONSE:')
print(response)

In [ ]:
result_text = f'''Generative AI Task 4 — Speech-to-Reasoning Result

Audio file: {audio_path}
Detected language: {info.language}

Whisper transcription:
{transcription}

Quantized LLM response:
{response}

Model: {MODEL_NAME}
GPU: {torch.cuda.get_device_name(0)}
'''

Path('speech_to_reasoning_output.txt').write_text(result_text, encoding='utf-8')
files.download('speech_to_reasoning_output.txt')
print('Result file created successfully!')